In [1]:
import pandas as pd 
import numpy as np 

In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

## *Multiple-Choice Data Formatting*
*In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.*

*Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4
What is the encoded numeric label for the row at index 150?*

In [3]:
label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train['encoded_answer'] = train['answer'].map(label_mapping)
encoded_label_150 = train.loc[150, 'encoded_answer']
print(encoded_label_150)

2


*Q2. Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)
What is the exact character length of this formatted input string?*


In [4]:
prompt = train.loc[0, 'prompt']
option_B = train.loc[0, 'B']

formatted_input = str(prompt) + " [SEP] " + str(option_B)
print(len(formatted_input))

407


## *Tokenization for Multiple-Choice Models*
*Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length*

*Since each question has five options, every row becomes five tokenized sequences.*

*Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"*

*After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]*

*What is the value of the second dimension?*


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
options = ['A', 'B', 'C', 'D', 'E']
prompt = train.loc[0, 'prompt']

formatted_inputs = [str(prompt) + " [SEP] " + str(train.loc[0, opt]) for opt in options]

encoding = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding['input_ids'].unsqueeze(0)
print(input_ids.shape[1])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

5


*Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.*

*The final input_ids tensor has shape:
[16, 5, 128]*

*How many total token positions are in this tensor?*

In [6]:
total_tokens = 16 * 5 * 128
print(total_tokens)

10240


## *Multiple-Choice Model Outputs*
*AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.*

*Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.*

*The output logits tensor has shape:
[1, 5]*

*How many logits are produced for one question?*

In [ ]:
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
outputs = model(input_ids=input_ids)
logits = outputs.logits

print(logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

*Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.*

*The model returns a scalar loss tensor.*

*How many dimensions does this loss tensor have?*


In [ ]:
import torch

label = torch.tensor([train.loc[0, 'encoded_answer']])
outputs = model(input_ids=input_ids, labels=label)
loss = outputs.loss

print(loss.ndim)

## *LoRA for Efficient Fine-Tuning*
*LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.*

*Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS*

*Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)*

*How many parameters are trainable?*

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, peft_config)
print(sum(p.numel() for p in model.parameters() if p.requires_grad))